In [104]:
import pm4py
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from datetime import datetime
from sklearn.metrics import confusion_matrix

import graphviz

print(graphviz.__version__)
import os

os.environ["PATH"] += os.pathsep + r"C:\Program Files\Graphviz\bin"



from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.ensemble import IsolationForest

from sklearn.linear_model import LogisticRegression






import warnings
warnings.filterwarnings("ignore")

0.21


In [2]:
# Read the XES event_log (BPI_challenge_2019)
log = pm4py.read_xes("BPI_Challenge_2019.xes")

# Convert to pandas DataFrame
df = pm4py.convert_to_dataframe(log)

# Export to CSV
df.to_csv("BPI_Challenge_2019.csv", index=False)

print("Conversion completed!")

parsing log, completed traces ::   0%|          | 0/251734 [00:00<?, ?it/s]

Conversion completed!


In [47]:
# Load data (BPI_challenge_2019)
bpi_challenge_2019 = pd.read_csv("BPI_Challenge_2019.csv")

In [48]:
#  Convert timestamp
bpi_challenge_2019['time:timestamp'] = pd.to_datetime(bpi_challenge_2019['time:timestamp'])
bpi_challenge_2019.sort_values('time:timestamp')
bpi_challenge_2019

,User,org:resource,concept:name,Cumulative net worth (EUR),time:timestamp,case:Spend area text,case:Company,case:Document Type,case:Sub spend area text,case:Purchasing Document,...,case:Vendor,case:Item Type,case:Item Category,case:Spend classification text,case:Source,case:Name,case:GR-Based Inv. Verif.,case:Item,case:concept:name,case:Goods Receipt
0,batch_00,batch_00,SRM: Created,298.0,2018-01-02 12:53:00+00:00,CAPEX & SOCS,companyID_0000,EC Purchase order,Facility Management,2000000000,...,vendorID_0000,Standard,"3-way match, invoice before GR",NPR,sourceSystemID_0000,vendor_0000,False,1,2000000000_00001,True
1,batch_00,batch_00,SRM: Complete,298.0,2018-01-02 13:53:00+00:00,CAPEX & SOCS,companyID_0000,EC Purchase order,Facility Management,2000000000,...,vendorID_0000,Standard,"3-way match, invoice before GR",NPR,sourceSystemID_0000,vendor_0000,False,1,2000000000_00001,True
2,batch_00,batch_00,SRM: Awaiting Approval,298.0,2018-01-02 13:53:00+00:00,CAPEX & SOCS,companyID_0000,EC Purchase order,Facility Management,2000000000,...,vendorID_0000,Standard,"3-way match, invoice before GR",NPR,sourceSystemID_0000,vendor_0000,False,1,2000000000_00001,True
3,batch_00,batch_00,SRM: Document Completed,298.0,2018-01-02 13:53:00+00:00,CAPEX & SOCS,companyID_0000,EC Purchase order,Facility Management,2000000000,...,vendorID_0000,Standard,"3-way match, invoice before GR",NPR,sourceSystemID_0000,vendor_0000,False,1,2000000000_00001,True
4,batch_00,batch_00,SRM: In Transfer to Execution Syst.,298.0,2018-01-02 13:53:00+00:00,CAPEX & SOCS,companyID_0000,EC Purchase order,Facility Management,2000000000,...,vendorID_0000,Standard,"3-way match, invoice before GR",NPR,sourceSystemID_0000,vendor_0000,False,1,2000000000_00001,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1595918,user_603,user_603,Change Approval for Purchase Order,1385.0,2019-01-17 14:00:00+00:00,NaN,companyID_0003,Framework order,NaN,4508076348,...,vendorID_1974,Limit,2-way match,NaN,sourceSystemID_0000,vendor_1898,False,90,4508076348_00090,False
1595919,user_602,user_602,Create Purchase Order Item,1385.0,2019-01-17 13:32:00+00:00,NaN,companyID_0003,Framework order,NaN,4508076348,...,vendorID_1974,Limit,2-way match,NaN,sourceSystemID_0000,vendor_1898,False,100,4508076348_00100,False
1595920,user_603,user_603,Change Approval for Purchase Order,1385.0,2019-01-17 14:00:00+00:00,NaN,companyID_0003,Framework order,NaN,4508076348,...,vendorID_1974,Limit,2-way match,NaN,sourceSystemID_0000,vendor_1898,False,100,4508076348_00100,False
1595921,user_602,user_602,Create Purchase Order Item,1385.0,2019-01-17 13:32:00+00:00,NaN,companyID_0003,Framework order,NaN,4508076348,...,vendorID_1974,Limit,2-way match,NaN,sourceSystemID_0000,vendor_1898,False,110,4508076348_00110,False


In [49]:
# Check the missing value
bpi_challenge_2019.isnull().sum()

User                                  0
org:resource                          0
concept:name                          0
Cumulative net worth (EUR)            0
time:timestamp                        0
case:Spend area text              16294
case:Company                          0
case:Document Type                    0
case:Sub spend area text          16294
case:Purchasing Document              0
case:Purch. Doc. Category name        0
case:Vendor                           0
case:Item Type                        0
case:Item Category                    0
case:Spend classification text    16294
case:Source                           0
case:Name                             0
case:GR-Based Inv. Verif.             0
case:Item                             0
case:concept:name                     0
case:Goods Receipt                    0
dtype: int64

In [31]:
# List of all columns
bpi_challenge_2019.columns

Index(['User', 'org:resource', 'concept:name', 'Cumulative net worth (EUR)',
       'time:timestamp', 'case:Spend area text', 'case:Company',
       'case:Document Type', 'case:Sub spend area text',
       'case:Purchasing Document', 'case:Purch. Doc. Category name',
       'case:Vendor', 'case:Item Type', 'case:Item Category',
       'case:Spend classification text', 'case:Source', 'case:Name',
       'case:GR-Based Inv. Verif.', 'case:Item', 'case:concept:name',
       'case:Goods Receipt'],
      dtype='object')

In [32]:
# Drop some of the columns
bpi_challenge_2019 = bpi_challenge_2019.drop(columns=['User', 'case:Spend area text', 'case:Company', 'case:Document Type', 
                                             'case:Sub spend area text', 'case:Purchasing Document', 'case:Purch. Doc. Category name',
                                             'case:Vendor', 'case:Item Type', 'case:Item Category', 'case:Spend classification text', 
                                             'case:Source', 'case:Name', 'case:GR-Based Inv. Verif.', 'case:Item', 'case:Goods Receipt'])

bpi_challenge_2019

,org:resource,concept:name,Cumulative net worth (EUR),time:timestamp,case:concept:name
0,batch_00,SRM: Created,298.0,2018-01-02 12:53:00+00:00,2000000000_00001
1,batch_00,SRM: Complete,298.0,2018-01-02 13:53:00+00:00,2000000000_00001
2,batch_00,SRM: Awaiting Approval,298.0,2018-01-02 13:53:00+00:00,2000000000_00001
3,batch_00,SRM: Document Completed,298.0,2018-01-02 13:53:00+00:00,2000000000_00001
4,batch_00,SRM: In Transfer to Execution Syst.,298.0,2018-01-02 13:53:00+00:00,2000000000_00001
...,...,...,...,...,...
1595918,user_603,Change Approval for Purchase Order,1385.0,2019-01-17 14:00:00+00:00,4508076348_00090
1595919,user_602,Create Purchase Order Item,1385.0,2019-01-17 13:32:00+00:00,4508076348_00100
1595920,user_603,Change Approval for Purchase Order,1385.0,2019-01-17 14:00:00+00:00,4508076348_00100
1595921,user_602,Create Purchase Order Item,1385.0,2019-01-17 13:32:00+00:00,4508076348_00110


In [33]:
# Check the missing value again 
bpi_challenge_2019.isnull().sum()

org:resource                  0
concept:name                  0
Cumulative net worth (EUR)    0
time:timestamp                0
case:concept:name             0
dtype: int64

In [34]:
# Check duplicates
bpi_challenge_2019.duplicated().sum()

np.int64(132748)

In [35]:
# Drop duplicates
bpi_challenge_2019 = bpi_challenge_2019.drop_duplicates()
bpi_challenge_2019

,org:resource,concept:name,Cumulative net worth (EUR),time:timestamp,case:concept:name
0,batch_00,SRM: Created,298.0,2018-01-02 12:53:00+00:00,2000000000_00001
1,batch_00,SRM: Complete,298.0,2018-01-02 13:53:00+00:00,2000000000_00001
2,batch_00,SRM: Awaiting Approval,298.0,2018-01-02 13:53:00+00:00,2000000000_00001
3,batch_00,SRM: Document Completed,298.0,2018-01-02 13:53:00+00:00,2000000000_00001
4,batch_00,SRM: In Transfer to Execution Syst.,298.0,2018-01-02 13:53:00+00:00,2000000000_00001
...,...,...,...,...,...
1595918,user_603,Change Approval for Purchase Order,1385.0,2019-01-17 14:00:00+00:00,4508076348_00090
1595919,user_602,Create Purchase Order Item,1385.0,2019-01-17 13:32:00+00:00,4508076348_00100
1595920,user_603,Change Approval for Purchase Order,1385.0,2019-01-17 14:00:00+00:00,4508076348_00100
1595921,user_602,Create Purchase Order Item,1385.0,2019-01-17 13:32:00+00:00,4508076348_00110


In [36]:
# Drop any NaN values
bpi_challenge_2019 = bpi_challenge_2019.dropna()
bpi_challenge_2019

,org:resource,concept:name,Cumulative net worth (EUR),time:timestamp,case:concept:name
0,batch_00,SRM: Created,298.0,2018-01-02 12:53:00+00:00,2000000000_00001
1,batch_00,SRM: Complete,298.0,2018-01-02 13:53:00+00:00,2000000000_00001
2,batch_00,SRM: Awaiting Approval,298.0,2018-01-02 13:53:00+00:00,2000000000_00001
3,batch_00,SRM: Document Completed,298.0,2018-01-02 13:53:00+00:00,2000000000_00001
4,batch_00,SRM: In Transfer to Execution Syst.,298.0,2018-01-02 13:53:00+00:00,2000000000_00001
...,...,...,...,...,...
1595918,user_603,Change Approval for Purchase Order,1385.0,2019-01-17 14:00:00+00:00,4508076348_00090
1595919,user_602,Create Purchase Order Item,1385.0,2019-01-17 13:32:00+00:00,4508076348_00100
1595920,user_603,Change Approval for Purchase Order,1385.0,2019-01-17 14:00:00+00:00,4508076348_00100
1595921,user_602,Create Purchase Order Item,1385.0,2019-01-17 13:32:00+00:00,4508076348_00110
